# Phase 2: MRI preprocessing and train/validation split

This notebook creates a reproducible, stratified split from **only** `data/raw/mri/Training`. The original `Testing` folder remains a final holdout and is never used to choose a preprocessing method.

The output is metadata rather than copied images: raw files stay immutable and later training can apply deterministic preprocessing on the fly.


## 1. Imports and configuration

A fixed seed makes the split repeatable; stratification preserves the four class proportions. `IMG_SIZE` is deliberately configurable, not a final architecture decision.


In [ ]:
from __future__ import annotations

from hashlib import sha256
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
VALIDATION_FRACTION = 0.20
IMG_SIZE = (224, 224)  # Change later only when the selected model requires it.
BACKGROUND_THRESHOLD = 10  # Pixel intensity used only for the exploratory margin crop.
CLASSES = ('glioma', 'meningioma', 'notumor', 'pituitary')
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

plt.rcParams['figure.dpi'] = 120


## 2. Project paths and Phase 1 inventory

The Phase 1 inventory is reused for audit fields such as SHA-256 hashes. Its recorded absolute paths may be stale after a folder rename, so paths are reconstructed from `split`, `class`, and `filename` and checked against the current project.


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'raw' / 'mri' / 'Training').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from the project or a subdirectory of it.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MRI_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'mri'
TRAINING_DIR = MRI_ROOT / 'Training'
TESTING_DIR = MRI_ROOT / 'Testing'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
METRICS_DIR = RESULTS_DIR / 'metrics'
INVENTORY_PATH = METRICS_DIR / 'mri_image_inventory.csv'
SPLIT_PATH = METRICS_DIR / 'mri_train_val_split.csv'
REPORT_PATH = METRICS_DIR / 'mri_phase2_report.txt'

assert INVENTORY_PATH.exists(), f'Missing Phase 1 inventory: {INVENTORY_PATH}'
assert TRAINING_DIR.is_dir() and TESTING_DIR.is_dir(), 'Expected Training and Testing directories.'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

inventory = pd.read_csv(INVENTORY_PATH)
inventory['path'] = inventory.apply(
    lambda row: MRI_ROOT / row['split'] / row['class'] / row['filename'], axis=1
)
print(f'Project root: {PROJECT_ROOT}')
print(f'Phase 1 inventory rows: {len(inventory):,}')
print(inventory['split'].value_counts().to_string())


## 3. Build metadata and create the split

Only the original Training rows participate. This prevents the holdout test set from influencing validation membership, preprocessing comparisons, or later model selection.


In [ ]:
def build_training_metadata(audit_inventory: pd.DataFrame) -> pd.DataFrame:
    columns = ['filename', 'class', 'width', 'height', 'aspect_ratio', 'is_square',
               'mode', 'dark_pixel_ratio', 'sha256', 'path']
    training = audit_inventory.loc[audit_inventory['split'].eq('Training'), columns].copy()
    training['path'] = training['path'].map(Path)
    missing = training.loc[~training['path'].map(Path.exists), 'path']
    if not missing.empty:
        raise FileNotFoundError(f'{len(missing)} training files from inventory are missing; first: {missing.iloc[0]}')
    training['relative_path'] = training['path'].map(lambda p: p.relative_to(PROJECT_ROOT).as_posix())
    if training['relative_path'].duplicated().any():
        raise ValueError('Training metadata contains duplicated file paths.')
    return training.reset_index(drop=True)

def stratified_split(metadata: pd.DataFrame) -> pd.DataFrame:
    train_ids, validation_ids = train_test_split(
        metadata.index.to_numpy(), test_size=VALIDATION_FRACTION,
        random_state=RANDOM_STATE, stratify=metadata['class'].to_numpy(), shuffle=True
    )
    split = metadata.copy()
    split['split'] = 'train'
    split.loc[validation_ids, 'split'] = 'validation'
    return split.sort_values(['split', 'class', 'relative_path']).reset_index(drop=True)

training_metadata = build_training_metadata(inventory)
split_metadata = stratified_split(training_metadata)
split_export = split_metadata[['relative_path', 'class', 'split', 'filename', 'sha256']].rename(columns={'relative_path': 'filepath'})
split_export.to_csv(SPLIT_PATH, index=False)
print(f'Saved {len(split_metadata):,} split rows to: {SPLIT_PATH}')


## 4. Split validation

We check class balance, assignment reproducibility, path overlap, hash overlap, and whether known internal exact-duplicate groups cross the new train/validation boundary. The last check is reported, not used to silently remove data.


In [ ]:
def directory_manifest(directory: Path) -> tuple[tuple[str, int, int], ...]:
    return tuple(sorted((p.relative_to(directory).as_posix(), p.stat().st_size, p.stat().st_mtime_ns)
                        for p in directory.rglob('*') if p.is_file()))

test_manifest_before = directory_manifest(TESTING_DIR)
class_counts = pd.crosstab(split_metadata['class'], split_metadata['split']).reindex(CLASSES)
class_counts['total'] = class_counts.sum(axis=1)
print('Class distribution:')
print(class_counts)
print('\nOverall:')
print(split_metadata['split'].value_counts().reindex(['train', 'validation']))

train_paths = set(split_metadata.loc[split_metadata['split'].eq('train'), 'relative_path'])
validation_paths = set(split_metadata.loc[split_metadata['split'].eq('validation'), 'relative_path'])
path_overlap = train_paths & validation_paths

repeat_split = stratified_split(training_metadata)
reproducible = split_metadata[['relative_path', 'split']].sort_values('relative_path').reset_index(drop=True).equals(
    repeat_split[['relative_path', 'split']].sort_values('relative_path').reset_index(drop=True)
)
hash_splits = split_metadata.groupby('sha256')['split'].nunique()
duplicate_hashes_crossing_split = int((hash_splits > 1).sum())
duplicate_files_crossing_split = int(split_metadata[split_metadata['sha256'].isin(hash_splits[hash_splits > 1].index)].shape[0])

assert not path_overlap, 'A file was assigned to both train and validation.'
assert reproducible, 'The same seed did not reproduce the same split.'
assert len(split_metadata) == 5600, 'Unexpected number of Training records.'
assert set(split_metadata['class']) == set(CLASSES), 'Unexpected classes in split.'
print(f'\nPath overlap: {len(path_overlap)}')
print(f'Reproducible with seed {RANDOM_STATE}: {reproducible}')
print(f'Known exact-duplicate SHA-256 groups crossing split: {duplicate_hashes_crossing_split}')
print(f'Files belonging to those cross-split duplicate groups: {duplicate_files_crossing_split}')


## 5. Deterministic preprocessing strategies

All strategies explicitly convert every image to RGB. The crop is exploratory: it detects a non-dark bounding box, adds a small context margin, and falls back to the original image when the candidate crop would be implausibly small. No raw file is written or changed.


In [ ]:
def load_rgb(path: Path) -> Image.Image:
    with Image.open(path) as image:
        return image.convert('RGB')

def direct_resize(image: Image.Image, size: tuple[int, int] = IMG_SIZE) -> Image.Image:
    return image.convert('RGB').resize(size, Image.Resampling.LANCZOS)

def resize_with_padding(image: Image.Image, size: tuple[int, int] = IMG_SIZE, fill=(0, 0, 0)) -> Image.Image:
    return ImageOps.pad(image.convert('RGB'), size, method=Image.Resampling.LANCZOS, color=fill, centering=(0.5, 0.5))

def crop_non_background(image: Image.Image, threshold: int = BACKGROUND_THRESHOLD, context_fraction: float = 0.05) -> tuple[Image.Image, tuple[int, int, int, int], bool]:
    rgb = image.convert('RGB')
    gray = np.asarray(rgb.convert('L'))
    foreground = gray > threshold
    if not foreground.any():
        return rgb, (0, 0, rgb.width, rgb.height), False
    ys, xs = np.where(foreground)
    left, right, top, bottom = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    margin_x, margin_y = round(rgb.width * context_fraction), round(rgb.height * context_fraction)
    box = (max(0, left - margin_x), max(0, top - margin_y), min(rgb.width, right + margin_x), min(rgb.height, bottom + margin_y))
    crop_fraction = ((box[2] - box[0]) * (box[3] - box[1])) / (rgb.width * rgb.height)
    if crop_fraction < 0.35:
        return rgb, (0, 0, rgb.width, rgb.height), False
    return rgb.crop(box), box, True

def crop_then_resize(image: Image.Image, size: tuple[int, int] = IMG_SIZE) -> tuple[Image.Image, tuple[int, int, int, int], bool]:
    cropped, box, applied = crop_non_background(image)
    return resize_with_padding(cropped, size=size), box, applied

def padding_fraction(original_size: tuple[int, int], size: tuple[int, int] = IMG_SIZE) -> float:
    width, height = original_size
    scale = min(size[0] / width, size[1] / height)
    content_area = (width * scale) * (height * scale)
    return 1 - content_area / (size[0] * size[1])


## 6. Representative visual comparison

One deterministic, aspect-ratio-challenging Training image per class is shown. These figures support inspection; they do **not** select a winner or use the final Testing set.


In [ ]:
def representative_rows(metadata: pd.DataFrame) -> pd.DataFrame:
    candidates = metadata.loc[metadata['split'].eq('train')].copy()
    candidates['shape_deviation'] = (np.log(candidates['width'] / candidates['height'])).abs()
    return (candidates.sort_values(['class', 'shape_deviation', 'relative_path'], ascending=[True, False, True])
            .groupby('class', sort=False).head(1).set_index('class').loc[list(CLASSES)].reset_index())

examples = representative_rows(split_metadata)
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(12, 12))
strategies = ('Original RGB', 'Direct resize', 'Preserve ratio + pad', 'Margin crop + pad')
for row_index, row in examples.reset_index(drop=True).iterrows():
    image = load_rgb(row['path'])
    cropped_result, crop_box, crop_applied = crop_then_resize(image)
    rendered = (image, direct_resize(image), resize_with_padding(image), cropped_result)
    for column, (title, rendered_image) in enumerate(zip(strategies, rendered)):
        axis = axes[row_index, column]
        axis.imshow(rendered_image)
        axis.axis('off')
        if row_index == 0:
            axis.set_title(title, fontsize=10)
    axes[row_index, 0].text(0.03, 0.04, f"{row['class']} ({image.size[0]}×{image.size[1]})", transform=axes[row_index, 0].transAxes, color='white', fontsize=8, bbox={'facecolor': 'black', 'alpha': 0.65, 'pad': 2})
    axes[row_index, 3].set_xlabel(f"crop {'applied' if crop_applied else 'fallback'}: {crop_box}", fontsize=8)
fig.suptitle('MRI preprocessing comparison — Training subset only', fontsize=14)
fig.tight_layout(rect=(0.10, 0, 1, 0.96))
comparison_path = FIGURES_DIR / 'mri_preprocessing_comparison.png'
fig.savefig(comparison_path, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {comparison_path}')


## 7. Background and crop-behaviour analysis

This describes how much content would change under padding/cropping and whether that behaviour differs by class. It is a shortcut-risk check, not a clinical claim and not evidence that cropping improves classification.


In [ ]:
def crop_metrics(row: pd.Series) -> dict[str, float | bool]:
    image = load_rgb(row['path'])
    _, box, applied = crop_non_background(image)
    crop_area = (box[2] - box[0]) * (box[3] - box[1])
    return {
        'crop_applied': applied,
        'crop_area_fraction': crop_area / (image.width * image.height),
        'padding_fraction': padding_fraction(image.size),
    }

analysis_rows = split_metadata.loc[split_metadata['split'].eq('train')].copy()
crop_summary = pd.DataFrame([crop_metrics(row) for _, row in analysis_rows.iterrows()])
analysis_rows = pd.concat([analysis_rows.reset_index(drop=True), crop_summary], axis=1)
background_summary = analysis_rows.groupby('class').agg(
    mean_dark_ratio=('dark_pixel_ratio', 'mean'),
    median_crop_area=('crop_area_fraction', 'median'),
    crop_applied_rate=('crop_applied', 'mean'),
    mean_padding_fraction=('padding_fraction', 'mean'),
).reindex(CLASSES)
display(background_summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
background_summary['mean_dark_ratio'].plot.bar(ax=axes[0], title='Mean dark-pixel ratio (Phase 1 metric)', ylim=(0, 1), color='#5271ff')
background_summary['median_crop_area'].plot.bar(ax=axes[1], title='Median retained area after margin crop', ylim=(0, 1), color='#00a896')
background_summary['mean_padding_fraction'].plot.bar(ax=axes[2], title='Mean padding fraction for aspect-preserving resize', ylim=(0, 1), color='#f4a261')
for axis in axes:
    axis.set_xlabel('Class')
    axis.tick_params(axis='x', rotation=30)
fig.suptitle('Preprocessing behaviour on the Training subset only')
fig.tight_layout()
background_path = FIGURES_DIR / 'mri_background_analysis.png'
fig.savefig(background_path, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {background_path}')


## 8. Output validation and Phase 2 report

The final checks reopen representative images through every strategy, confirm 224×224 RGB output, and compare the Testing-directory manifest captured before and after the notebook. Dataset-level mean/std normalization is intentionally deferred: if needed later, it must be calculated from the Training subset only or replaced by the selected pretrained model’s documented preprocessing.


In [ ]:
def validate_outputs(examples: pd.DataFrame) -> None:
    for path in examples['path']:
        image = load_rgb(path)
        outputs = (direct_resize(image), resize_with_padding(image), crop_then_resize(image)[0])
        for output in outputs:
            assert output.mode == 'RGB', f'Unexpected mode: {output.mode}'
            assert output.size == IMG_SIZE, f'Unexpected size: {output.size}'

validate_outputs(examples)
test_manifest_after = directory_manifest(TESTING_DIR)
testing_modified_during_notebook = test_manifest_before != test_manifest_after
assert not testing_modified_during_notebook, 'Testing directory changed while this notebook ran.'

counts_text = class_counts.to_string()
report = f"""MRI PHASE 2 — PREPROCESSING & SPLIT
====================================

Dataset:
Training images: {len(split_metadata)}
Testing images: {len(test_manifest_before)}

Train/Validation split:
Train: {(split_metadata['split'] == 'train').sum()}
Validation: {(split_metadata['split'] == 'validation').sum()}
Random seed: {RANDOM_STATE}
Stratified: YES

Class distribution:
{counts_text}

Validation checks:
Train ↔ Validation path overlap: {len(path_overlap)}
Same-seed split reproducible: {reproducible}
Known exact-duplicate SHA-256 groups crossing Train/Validation: {duplicate_hashes_crossing_split}
Files in those groups: {duplicate_files_crossing_split}
Testing modified during notebook execution: {'YES' if testing_modified_during_notebook else 'NO'}

Image preprocessing:
RGB conversion: YES (Pillow convert('RGB') for every loaded image)
Target size: {IMG_SIZE} (configurable; not an architecture decision)
Strategies inspected: direct resize; aspect-ratio-preserving resize + black padding; exploratory margin crop + aspect-preserving resize + padding.

Preprocessing comparison:
The comparison is visual and descriptive only; no model was trained and no test image was used. Direct resize can geometrically distort non-square images. Padding preserves geometry but can retain class-correlated border/background characteristics. The exploratory crop reduces some borders but changes retained context by image/class, so it is not selected automatically.

Final preprocessing recommendation:
Keep direct resize and aspect-ratio-preserving padding as candidate deterministic pipelines for Phase 3/4 validation experiments. Do not adopt margin cropping as the default until a validation-only model comparison shows that it helps without creating unacceptable shortcut risk.

Known limitations:
The original predefined Testing split has substantial perceptual/structural similarity to Training, while patient-level leakage cannot be confirmed because identifiers are unavailable. The new random Train/Validation split may also place known exact duplicates across subsets; these are reported above and were not automatically removed.

Ready for Phase 3: YES — after review of the figures and split policy.
"""
REPORT_PATH.write_text(report, encoding='utf-8')
print(report)
print(f'Report saved: {REPORT_PATH}')


## Phase 2 decision notes

- **Why metadata, not processed copies?** It avoids duplicating 5,600 raw images, keeps GitHub small, and makes future preprocessing changes traceable.
- **Why split before learned statistics?** Any dataset mean/std must come from `train` only, otherwise validation information leaks into the pipeline.
- **Why no augmentation here?** Random augmentation belongs to training only. Validation and final testing must use deterministic transforms.
- **Why no preprocessing winner yet?** A visual check detects obvious distortion or lost anatomy, but only later validation experiments can compare candidate pipelines. The final Testing set remains untouched until Phase 5.
